# GuacaMol Dataset Tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/instadeepai/alf/blob/main/tutorials/datasets/guacamol_tutorial.ipynb)

_Open in Colab works once ALF is public / on PyPI; until then, use the local setup below._

[GuacaMol](https://github.com/BenevolentAI/guacamol) (Brown et al., 2019) is a benchmark dataset
derived from ChEMBL containing ~1.6 million drug-like SMILES strings. Each molecule has 10 RDKit
physicochemical properties: `BertzCT`, `MolLogP`, `MolWt`, `TPSA`, `NumHAcceptors`, `NumHDonors`,
`NumRotatableBonds`, `NumAliphaticRings`, `NumAromaticRings`, and `QED`.

This tutorial demonstrates the `GuacaMol` dataset class from `alf_tools`:
1. Downloading and inspecting the raw SMILES files
2. Loading the dataset and building a pandas DataFrame
3. Visualising property distributions and correlations
4. Querying properties for arbitrary molecules
5. Case study: aspirin's physicochemical profile

We use `max_molecules=10_000` throughout so the notebook runs in under a minute on CPU.

## Setup

Run the cell below to install ALF and this tutorial's dependencies — **no repository clone required**, so it works in a fresh environment or on Google Colab.

- Already set up a dev environment from a clone (`uv sync`)? You can **skip the install cell**.
- To run on a **GPU**, uncomment the GPU line in the install cell.

For all installation options, see the [Installation Guide](https://instadeepai.github.io/alf/installation.html).
- On **Colab**, the first install can take a few minutes; if prompted, choose *Runtime ▸ Restart session* and re-run the cell. To use a GPU, set *Runtime ▸ Change runtime type ▸ GPU* and uncomment the GPU line in the install cell.

In [ ]:
# Install ALF + this tutorial's dependencies — no clone needed.
# (Skip this cell if you are already running from a cloned repo via `uv sync`.)
# TODO(pypi): once ALF is published to PyPI, replace the git install below with:
#   %pip install "alf_tools[guacamol]" matplotlib pandas seaborn
%pip install "alf_core @ git+https://github.com/instadeepai/alf.git#subdirectory=core" "alf_tools[guacamol] @ git+https://github.com/instadeepai/alf.git#subdirectory=tools" matplotlib pandas seaborn
# GPU (optional): run this AFTER the line above to switch PyTorch to a CUDA build.
# %pip install torch --index-url https://download.pytorch.org/whl/cu128

Import the plotting, data and chemistry libraries used throughout, then bring in the `GuacaMol` dataset class and its helper utilities from `alf_tools`. RDKit's deprecation warnings are silenced only around the `alf_tools` import — where the benchmark fingerprints are precomputed — and re-enabled immediately afterwards so genuine warnings such as invalid SMILES still surface.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from alf_core.dataclasses.candidate import Candidate, Modality
from rdkit import Chem, RDLogger

# `alf_tools` precomputes GuacaMol benchmark fingerprints at import time using RDKit's
# legacy fingerprint API, which emits "please use MorganGenerator/AtomPairGenerator"
# deprecation notices. Silence only RDKit's warning stream around the import, then
# re-enable it so genuine warnings (e.g. invalid SMILES) are still surfaced below.
RDLogger.DisableLog("rdApp.warning")
from alf_tools.datasets.guacamol.guacamol_dataset import GuacaMol, GuacaMolConfig  # noqa: E402
from alf_tools.datasets.guacamol.guacamol_utils import (  # noqa: E402
    ALL_PROPERTIES,
    DATAPATH,
    GUACAMOL_FILES,
    _cache_path,  # noqa: PLC2701
    _compute_properties,  # noqa: PLC2701
    download_guacamol,
)

RDLogger.EnableLog("rdApp.warning")
from rdkit.Chem.Draw import MolToImage  # noqa: E402

MAX_LINES = 10_000
PROPERTY_COLS = sorted(ALL_PROPERTIES)

print("✓ Imports OK")
print(f"Properties ({len(PROPERTY_COLS)}): {PROPERTY_COLS}")

## Section 1 — Download

`download_guacamol` streams all four GuacaMol SMILES files from Figshare and caches them locally.
With `max_lines=10_000`, only the first 10 000 lines of each file are written to disk; subsequent
runs detect the existing files and skip the download entirely.

In [ ]:
download_guacamol(data_dir=DATAPATH, max_lines=MAX_LINES)

print("Downloaded files:")
for split, info in GUACAMOL_FILES.items():
    filepath = _cache_path(DATAPATH / info["name"], MAX_LINES)
    size_kb = filepath.stat().st_size / 1024
    print(f"  {split:5s}  {filepath.name}  ({size_kb:.1f} KB)")

Read the first five lines of the cached corpus to confirm the download worked and to show what the raw data looks like: one SMILES string per line. This is the unprocessed input the dataset class parses and computes properties from.

In [ ]:
all_smiles_path = _cache_path(DATAPATH / GUACAMOL_FILES["ALL"]["name"], MAX_LINES)
with open(all_smiles_path, encoding="utf-8") as f:
    sample = [next(f).strip() for _ in range(5)]

print("First 5 SMILES from the corpus:")
for i, smi in enumerate(sample, 1):
    print(f"  {i}. {smi}")

## Section 2 — Load & Build DataFrame

We instantiate `GuacaMolConfig` with `target_property="QED"` and `max_molecules=10_000`, then
`GuacaMol` loads the corpus and computes all 10 RDKit properties for each molecule.

Computed property values are stored in a single NumPy array `dataset._prop_matrix` (shape `(N, P)`)
rather than in individual `Candidate.features` dicts — `candidate.features` is always `{}` for
corpus molecules. The public `properties_dataframe()` accessor returns a pandas DataFrame aligned
with `_raw_dataset.candidates`.

In [ ]:
config = GuacaMolConfig(
    name="guacamol_tutorial",
    modality=Modality.SEQUENCE,
    seed=42,
    train_ratio=0.8,
    validation_frac=0.1,
    test_ratio=0.1,
    target_property="QED",
    computed_properties=PROPERTY_COLS,
    max_molecules=10_000,
    data_dir=DATAPATH,
)
dataset = GuacaMol(config)

n = len(dataset._raw_dataset.candidates)
print(dataset)
print(f"Loaded {n} candidates")

Build a pandas DataFrame via `properties_dataframe()` — keeping the `smiles` column plus the 10 property columns — and call `describe()` to summarise each property's range, mean and spread. This gives a first quantitative feel for the drug-like chemical space the corpus covers.

In [ ]:
df = dataset.properties_dataframe()[["smiles"] + PROPERTY_COLS]
print(f"Shape: {df.shape}")
df.describe()

## Section 3 — Dataset Statistics

We examine the distribution of SMILES string lengths, all 10 physicochemical properties
individually, and their pairwise correlations.

In [ ]:
df["smiles_len"] = df["smiles"].str.len()

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df["smiles_len"], kde=True, ax=ax)
ax.set_xlabel("SMILES string length (characters)")
ax.set_title("Distribution of SMILES Lengths (n=10 000)")
plt.tight_layout()
plt.show()

Most SMILES strings are short — typically a few dozen characters — with a long right tail from a minority of larger, more complex molecules. This skew reflects the drug-like bias of the ChEMBL-derived corpus, where small-to-medium molecules dominate.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for ax, prop in zip(axes.flat, PROPERTY_COLS):
    sns.histplot(df[prop], kde=True, ax=ax)
    ax.set_title(prop, fontsize=11)
    ax.set_xlabel("")
fig.suptitle("Distribution of RDKit Physicochemical Properties (n=10 000)", fontsize=14)
plt.tight_layout()
plt.show()

Each panel shows one property's distribution across the 10 000 molecules. Properties such as `MolWt`, `MolLogP` and `TPSA` are roughly bell-shaped around typical drug-like values, while count-based properties (`NumHDonors`, `NumAromaticRings`, `NumAliphaticRings`) are discrete and concentrated on small integers. `QED` clusters towards higher values, consistent with a corpus of drug-like molecules.

In [ ]:
corr = df[PROPERTY_COLS].corr()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    ax=ax,
)
ax.set_title("Property Correlation Matrix", fontsize=13)
plt.tight_layout()
plt.show()

Warm cells are positively correlated descriptors, cool cells negatively correlated, and values near zero (pale) are largely independent. Size-related properties — molecular weight, heavy-atom and ring counts — tend to form a positively correlated block, while lipophilicity (`MolLogP`) and polar surface area (`TPSA`) typically pull in opposite directions. Strongly correlated descriptors carry redundant information, which is worth knowing before using them as model features.

## Section 4 — Querying

`dataset.query()` accepts a list of `Candidate` objects and returns a `LabelledCandidates` with
the `target_property` label for each molecule. For SMILES already in the corpus the stored label
is returned directly; for novel molecules RDKit computes it on the fly.

**Note:** `query()` returns only the `target_property` label (`QED` here) — not all 10 properties.
See Section 5 for how to retrieve the full property profile for a single molecule.

In [ ]:
QUERY_MOLECULES = {
    "aspirin": "CC(=O)Oc1ccccc1C(=O)O",
    "ibuprofen": "CC(C)Cc1ccc(cc1)C(C)C(=O)O",
    "caffeine": "Cn1cnc2c1c(=O)n(c(=O)n2C)C",
}

query_candidates = [
    Candidate(data=smi, modality=Modality.SEQUENCE) for smi in QUERY_MOLECULES.values()
]

results = dataset.query(query_candidates)

print(f"Return type : {type(results).__name__}")
print(f"Labels shape: {results.labels.shape}")
print("\nFirst result:")
print(f"  SMILES : {results.candidates[0].data}")
print(f"  QED    : {results.labels[0]:.4f}")

Tabulate the query results so the per-molecule `QED` labels are easy to read side by side. Each row pairs a molecule name with its SMILES and the single `target_property` label that `query()` returned.

In [ ]:
query_rows = [
    {"molecule": name, "smiles": cand.data, "QED (label)": float(label)}
    for name, cand, label in zip(QUERY_MOLECULES.keys(), results.candidates, results.labels)
]
pd.DataFrame(query_rows).set_index("molecule")

## Section 5 — Aspirin Case Study

Aspirin (acetylsalicylic acid) is a well-characterised drug useful as a concrete reference point.
We:
1. Draw its 2D molecular structure inline using RDKit.
2. Compute all 10 physicochemical properties directly via `_compute_properties` — necessary
   because `query()` only returns the `target_property` label for novel SMILES not in the corpus.
3. Compare aspirin's profile to the dataset via a z-score bar chart and a raw values table.
4. Overlay aspirin's values on the Section 3 property histograms.

In [ ]:
ASPIRIN_SMILES = "CC(=O)Oc1ccccc1C(=O)O"

mol = Chem.MolFromSmiles(ASPIRIN_SMILES)
img = MolToImage(mol, size=(300, 300))

fig, ax = plt.subplots(figsize=(3, 3))
ax.imshow(img)
ax.axis("off")
ax.set_title("Aspirin\nCC(=O)Oc1ccccc1C(=O)O", fontsize=9)
plt.tight_layout()
plt.show()

The rendered 2D structure confirms RDKit parsed the aspirin SMILES correctly: a benzene ring bearing an acetyl ester and a carboxylic acid group. This is the reference molecule used for the property comparisons that follow.

Compute all 10 physicochemical properties for aspirin directly with `_compute_properties`. This is needed because `query()` only returns the `target_property` label (`QED`), whereas the comparisons below require aspirin's full property profile.

In [ ]:
aspirin_props = _compute_properties(ASPIRIN_SMILES, PROPERTY_COLS)

aspirin_raw = pd.Series(aspirin_props, name="aspirin")[PROPERTY_COLS]
print("Aspirin — raw property values:")
print(aspirin_raw.round(3).to_string())

Standardise aspirin's properties into z-scores — how many standard deviations each value lies from the dataset mean — and plot them as a bar chart. Standardising puts properties on different natural scales (e.g. `MolWt` versus `NumHDonors`) onto a common axis so aspirin's profile can be read at a glance.

In [ ]:
means = df[PROPERTY_COLS].mean()
stds = df[PROPERTY_COLS].std()

aspirin_z = pd.Series(
    {prop: (aspirin_props[prop] - means[prop]) / stds[prop] for prop in PROPERTY_COLS},
    name="z-score",
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x=list(aspirin_z.index), y=list(aspirin_z.values), ax=ax)
ax.axhline(0, color="black", linewidth=0.9, linestyle="--", label="dataset mean")
ax.set_xlabel("Property")
ax.set_ylabel("Z-score (σ from dataset mean)")
ax.set_title("Aspirin: Standardised Property Profile vs Dataset Mean (n=10 000)")
ax.tick_params(axis="x", rotation=45)
ax.legend()
plt.tight_layout()
plt.show()

Each bar is aspirin's value for a property expressed in standard deviations from the dataset mean (the dashed line at zero). Bars well below zero mark properties where aspirin sits at the lighter, simpler end of the chemical space — for example a low molecular weight and few rings — which is expected given aspirin is much smaller than a typical molecule in this drug-like corpus.

Assemble a side-by-side table of aspirin's raw values against the dataset mean, standard deviation and z-score for every property. This makes the standardised comparison from the bar chart legible as exact numbers.

In [ ]:
comparison = pd.DataFrame({
    "aspirin": aspirin_raw,
    "dataset_mean": means.round(3),
    "dataset_std": stds.round(3),
    "z_score": aspirin_z.round(3),
})
comparison

Overlay aspirin's value (red dashed line) on each property's distribution from Section 3. This locates aspirin within the full corpus visually, showing for each property whether it sits near the bulk of the data or out in a tail.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for i, (ax, prop) in enumerate(zip(axes.flat, PROPERTY_COLS)):
    sns.histplot(df[prop], kde=True, ax=ax, alpha=0.6)
    ax.axvline(
        aspirin_props[prop],
        color="red",
        linewidth=2,
        linestyle="--",
        label="aspirin" if i == 0 else None,
    )
    ax.set_title(prop, fontsize=11)
    ax.set_xlabel("")
axes.flat[0].legend(loc="upper right")
fig.suptitle(
    "Property Distributions with Aspirin Highlighted (red dashed line)",
    fontsize=14,
)
plt.tight_layout()
plt.show()

Aspirin (red dashed line) sits toward the low end of most size and complexity descriptors — it is a small, simple molecule relative to the GuacaMol corpus — which lines up with the negative z-scores in the bar chart above. Seeing where a molecule falls across every property at once is a quick sanity check before using it as a query or a seed for design.

## Summary

This tutorial demonstrated the `GuacaMol` dataset class:
- **Download** — stream and cache up to 10 000 SMILES from Figshare
- **Load** — `GuacaMol` computes 10 RDKit properties per molecule
- **Statistics** — SMILES lengths, property histograms, correlation heatmap
- **Query** — `dataset.query()` returns `target_property` labels for arbitrary SMILES
- **Case study** — aspirin sits at the lighter, simpler end of the drug-like chemical space

Next steps: try `target_property="MolWt"` or substitute a custom SMILES library.